In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5,),
        (0.5,)
    )
])

In [42]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [43]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [44]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x

In [45]:
model = SimpleCNN().to(device)

In [46]:
criterion = nn.CrossEntropyLoss()

In [47]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [48]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    train_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.2f}%")

Epoch [1/5]
Train Loss: 230.0753
Train Accuracy: 92.53%
Epoch [2/5]
Train Loss: 82.9934
Train Accuracy: 97.41%
Epoch [3/5]
Train Loss: 63.2584
Train Accuracy: 98.02%
Epoch [4/5]
Train Loss: 52.1053
Train Accuracy: 98.39%
Epoch [5/5]
Train Loss: 42.5420
Train Accuracy: 98.62%


In [58]:
model.eval()
correct = 0
total = 0
test_loss = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

test_accuracy = 100 * correct / total
adam_accuracy = test_accuracy
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy1: {test_accuracy:.2f}%")


Test Loss: 3.9703
Test Accuracy1: 99.15%


In [59]:
model = SimpleCNN().to(device)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)

In [60]:
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.2f}%")

Epoch [1/5]
Train Loss: 311.6147
Train Accuracy: 89.61%
Epoch [2/5]
Train Loss: 88.5897
Train Accuracy: 97.18%
Epoch [3/5]
Train Loss: 66.7349
Train Accuracy: 97.92%
Epoch [4/5]
Train Loss: 52.8913
Train Accuracy: 98.34%
Epoch [5/5]
Train Loss: 45.2539
Train Accuracy: 98.56%


In [61]:
model.eval()

correct = 0
total = 0
test_loss = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

sgd_accuracy  = 100 * correct / total
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {sgd_accuracy:.2f}%")


Test Loss: 4.5432
Test Accuracy: 99.05%


In [62]:
print(f"Adam Accuracy: {adam_accuracy:.2f}%")
print(f"SGD Accuracy: {sgd_accuracy:.2f}%")

Adam Accuracy: 99.15%
SGD Accuracy: 99.05%
